# Academic Question Answering System using RAG — V1

This notebook implements the complete V1 Retrieval-Augmented Generation pipeline for the Operating Systems notes PDF.



## 1. Project Configuration

In [18]:
# Install the packages required by this notebook.
%pip install langchain-community==0.4.2 pypdf==6.10.2 langchain-text-splitters requests python-dotenv pinecone google-genai


Note: you may need to restart the kernel to use updated packages.


In [19]:
# Imports and environment configuration
import os
import time
import requests

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from google import genai

load_dotenv()

JINA_API_KEY = os.getenv("JINA_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print("Jina:", JINA_API_KEY is not None)
print("Pinecone:", PINECONE_API_KEY is not None)
print("Gemini:", GEMINI_API_KEY is not None)


Jina: True
Pinecone: True
Gemini: True


## 2. PDF Ingestion & Text Chunking

The source document is loaded page-by-page and split into overlapping chunks for semantic retrieval.


In [20]:
PDF_PATH = "os_notes.PDF"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Pages:", len(documents))
print("Total chunks:", len(chunks))


Pages: 6
Total chunks: 26


In [21]:
# Optional inspection of extracted chunks
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i + 1} ---")
    print(chunk.page_content)
    print()


--- Chunk 1 ---
OPERATING SYSTEMS — COMPLETE CONCISE NOTES
PAGE 1 — OS FUNDAMENTALS & PROCESSES
1. Operating System
An OS is system software that manages hardware/resources and provides services to application programs. Main goals:
convenience, efficient resource use, and protection.
2. OS Services & Components
 Program execution, I/O operations, file-system manipulation, communication, error detection, resource allocation, accounting and
 protection/security.

--- Chunk 2 ---
protection/security.
 Kernel: core of the OS. System call: controlled interface through which programs request kernel services.
 
 OS structures: monolithic, layered, microkernel, modular and hybrid approaches.
 
3. Processes
 Process = program in execution. PCB stores process state, program counter, registers, scheduling and memory information.
 
 States: New → Ready → Running → Waiting/Blocked → Terminated. Context switch saves one process state and loads another.

--- Chunk 3 ---
 Process creation uses p

## 3. Jina Embeddings

Each chunk is converted into a vector using the hosted Jina embedding model.


In [22]:
EMBEDDING_MODEL = "jina-embeddings-v5-text-small"

texts = [chunk.page_content for chunk in chunks]
vectors = []

batch_size = 50

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]

    print(f"Embedding {i + 1} to {i + len(batch)} of {len(texts)}")

    response = requests.post(
        "https://api.jina.ai/v1/embeddings",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {JINA_API_KEY}"
        },
        json={
            "model": EMBEDDING_MODEL,
            "input": batch,
            "normalized": True
        }
    )

    response.raise_for_status()
    data = response.json()

    vectors.extend(
        [item["embedding"] for item in data["data"]]
    )

    time.sleep(0.2)

print("\nDone!")
print("Total chunks:", len(chunks))
print("Total vectors:", len(vectors))
print("Embedding dimension:", len(vectors[0]))


Embedding 1 to 26 of 26

Done!
Total chunks: 26
Total vectors: 26
Embedding dimension: 1024


## 4. Pinecone Vector Database

Vectors and metadata are stored in the `academic-rag` Pinecone serverless index using cosine similarity.


In [23]:
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "academic-rag"
dimension = len(vectors[0])

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec={
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        }
    )

index = pc.Index(index_name)

print("Connected to index:", index_name)
print("Vector dimension:", dimension)


Connected to index: academic-rag
Vector dimension: 1024


In [24]:
# Prepare Pinecone records with source and subject metadata.
records = []

for i, (chunk, vector) in enumerate(zip(chunks, vectors)):
    records.append({
        "id": f"os-chunk-{i}",
        "values": vector,
        "metadata": {
            "text": chunk.page_content,
            "page": chunk.metadata.get("page", 0),
            "source": "operating_systems_complete_concise_notes_6_pages.pdf",
            "subject": "Operating Systems"
        }
    })

print("Records prepared:", len(records))


Records prepared: 26


In [25]:
# Upload records in batches.
batch_size = 100

for i in range(0, len(records), batch_size):
    batch = records[i:i + batch_size]
    index.upsert(vectors=batch)
    print(f"Uploaded {i + len(batch)}/{len(records)}")


Uploaded 26/26


In [26]:
# Verify the Pinecone index.
stats = index.describe_index_stats()
print(stats)


DescribeIndexStatsResponse(dimension=1024, total_vector_count=257, metric='cosine', namespaces=1)


## 5. Query Embedding & Retrieval

A user question is embedded with the same Jina model and matched against the vectors stored in Pinecone.


In [27]:
def get_query_embedding(question):
    response = requests.post(
        "https://api.jina.ai/v1/embeddings",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {JINA_API_KEY}"
        },
        json={
            "model": EMBEDDING_MODEL,
            "input": [question],
            "normalized": True
        }
    )

    response.raise_for_status()
    return response.json()["data"][0]["embedding"]


In [28]:
def retrieve_chunks(question, top_k=15):
    query_vector = get_query_embedding(question)

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    retrieved_chunks = []

    for match in results["matches"]:
        retrieved_chunks.append({
            "text": match["metadata"]["text"],
            "page": match["metadata"].get("page", "Unknown"),
            "score": match["score"]
        })

    return retrieved_chunks


## 6. Adaptive Top-K Retrieval

Focused questions retrieve fewer chunks, while broad or conceptual questions retrieve more context.


In [29]:
def choose_top_k(question):
    question_lower = question.lower()

    broad_keywords = [
        "complete",
        "entire",
        "all topics",
        "all units",
        "full syllabus",
        "complete syllabus",
        "whole document",
        "everything",
        "overview"
    ]

    conceptual_keywords = [
        "explain",
        "difference",
        "compare",
        "how does",
        "why",
        "describe",
        "discuss"
    ]

    if any(keyword in question_lower for keyword in broad_keywords):
        return 50

    if any(keyword in question_lower for keyword in conceptual_keywords):
        return 10

    return 5


def smart_retrieve(question):
    top_k = choose_top_k(question)
    print(f"Retrieving top {top_k} chunks...")

    return retrieve_chunks(question, top_k=top_k)


## 7. Gemini Answer Generation

The retrieved chunks are placed into a grounded prompt. Gemini is instructed to answer only from the supplied context and not invent missing information.


In [30]:
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client ready!")


Gemini client ready!


In [31]:
def generate_answer(question, retrieved_chunks):
    context = ""

    for i, chunk in enumerate(retrieved_chunks):
        context += f"""
--- Context {i + 1} ---
Page: {chunk['page']}
{chunk['text']}
"""

    prompt = f"""
You are an academic syllabus question-answering assistant.

Use ONLY the information explicitly present in the provided context.

For syllabus questions:
- Do not infer or guess missing units.
- Do not use general knowledge.
- Do not derive topics from lab exercises, course outcomes, or related subjects unless they are explicitly listed as syllabus topics.
- If a unit is not present in the retrieved context, clearly state that the unit could not be found.
- Never invent missing syllabus content.

For a request for the COMPLETE syllabus:
- Include every unit that is explicitly present in the context.
- Preserve the unit numbers and topic structure.
- Do not claim that the syllabus is complete if one or more requested units are missing.

If the requested information cannot be found, say:
"I couldn't find the answer in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    max_retries = 3

    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )
            return response.text

        except Exception as e:
            print(f"Attempt {attempt + 1} failed:")
            print(e)

            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                raise


## 8. End-to-End RAG Test

Enter any question about the indexed Operating Systems document.


In [34]:
question = input("Ask a question: ")

retrieved_chunks = smart_retrieve(question)

answer = generate_answer(
    question,
    retrieved_chunks
)

print("\n" + "=" * 80)
print("ANSWER:")
print(answer)


Retrieving top 5 chunks...

ANSWER:
Based on the provided context, the Banker's Algorithm:
- Uses Available, Max, Allocation, and Need matrices, where $\text{Need} = \text{Max} - \text{Allocation}$.
- Uses a safety algorithm that repeatedly finds a process whose $\text{Need} \le \text{Work}$, assumes it finishes, and then adds its $\text{Allocation}$ to $\text{Work}$ ($\text{Work} = \text{Work} + \text{Allocation}$).
- Determines if a state is safe (if all processes can finish) or unsafe (otherwise).
